# Dynamic Conditional Correlation Modelling 

*By Vlad Popa*

Squared returns of assets act as a proxy for time-varying variance, revealing volatility clustering: the variability of returns in high-volatility periods tend to cluster together, and low-volatility periods follow one another, rather than occurring randomly. This proves that market volatility is not entirely random, and plays a significant role in risk management, derivative pricing and algorithmic trading.

In this project, we assess the time-varying, conditional correlation between several assets, driven by geopolitical choke points in 2026 using the DCC-GARCH(1,1) model. 

## 1. Import Libraries and Load Data

In [36]:
# Import libraries
library(rugarch)
library(rmgarch)
library(purrr)

# Read .csv files
bdi <- read.csv("input/mod/Modified Baltic Dry Index Historical Data.csv")
oil <- read.csv("input/mod/Modified Crude Oil WTI Futures Historical Data.csv")
soxx <- read.csv("input/mod/Modified SOXX ETF Stock Price History.csv")
urth <- read.csv("input/mod/Modified URTH ETF Stock Price History.csv")
gold <- read.csv("input/mod/Modified XAU_USD Historical Data.csv")

In [37]:
# Save returns of each asset to a separate dataframe
bdi_rets <- data.frame(Date = bdi$Date,  BDI = bdi$Return / 100)
oil_rets <- data.frame(Date = oil$Date,  OIL = oil$Return / 100)
soxx_rets <- data.frame(Date = soxx$Date, SOXX = soxx$Return / 100)
urth_rets <- data.frame(Date = urth$Date, URTH = urth$Return / 100)
gold_rets <- data.frame(Date = gold$Date, GOLD = gold$Return / 100)

# Save all dataframes to a list
rets_list <- list(bdi_rets, oil_rets, soxx_rets, urth_rets, gold_rets)

# Sequentially merge each dataframe on the 'Date' column
returns_scaled <- purrr::reduce(rets_list, merge, by = "Date")
rownames(returns_scaled) <- returns_scaled$Date
returns_scaled$Date <- NULL

## 2. GARCH Modelling

In [38]:
# Function for modelling conditional volatilities using eGARCH
my_garch <- function(returns_scaled){
    # Specify our GARCH model: how do individual variances behave?
    uspec <- ugarchspec(
        mean.model = list(armaOrder = c(0,0), include.mean = FALSE),
        variance.model = list(model = "sGARCH"),
        distribution.model = "norm")

    # Replicate the univariate GARCH model across all assets
    mspec <- multispec(replicate(ncol(returns_scaled), uspec))       

    # Estimate all conditional volatilities using our defined GARCH model
    fitlist <- multifit(multispec = mspec, data = returns_scaled)
    sigmas <- sigma(fitlist)
    rownames(sigmas) <- rownames(returns_scaled)
    colnames(sigmas) <- colnames(returns_scaled)
    
    # Save specifications, fits and conditional volatilities together
    output = list(
        sigmas = sigmas,
        mspec = mspec,
        fitlist = fitlist
    )
    return (output)
}

# Run function and save conditional volatilites
ugarch_results <- my_garch(returns_scaled) 
sigmas <- ugarch_results[["sigmas"]]
mspec <- ugarch_results[["mspec"]]
fitlist <- ugarch_results[["fitlist"]]
head(ugarch_results[["sigmas"]])

,BDI,OIL,SOXX,URTH,GOLD
2026-01-02,0.02765502,0.04374308,0.02827338,0.009258129,0.02091431
2026-01-05,0.02759484,0.04051297,0.02899411,0.009264485,0.01860048
2026-01-06,0.02753492,0.03802480,0.02871115,0.009270830,0.02133191
2026-01-07,0.02747506,0.03631624,0.02898010,0.009277164,0.01952609
2026-01-08,0.02741566,0.03460564,0.02864478,0.009283488,0.01814090
2026-01-09,0.02735648,0.03484459,0.02841817,0.009289801,0.01672695


## 3. DCC Modelling

In [39]:
# Function for modelling dynamic correlations between assets
my_dcc <- function(returns_scaled, mspec, fitlist){
    # Specify our DCC model: how do the correlations behave?
    dcc_spec <- dccspec(uspec = mspec, dccOrder = c(1,1), model = "DCC")
    dcc_fit <- dccfit(spec = dcc_spec, data = returns_scaled, fit = fitlist)
    corr_matrix <- rcor(dcc_fit)
    return (corr_matrix)
}

# Run function and save conditional correlation matrix
corr_matrix <- my_dcc(returns_scaled, mspec, fitlist) 